# Rate Allocator — interactive demo

Pick which institutions to include, set the total in MXN, and inspect principal, **overnight-annualized nominal rates** (see `data/sample1.yaml`), and **horizon-based** expected returns (compound gross minus modeled fees and estimated ISR over real interest). The time-series chart compares **365× discrete compounding** vs simple linear accrual on the same final allocation.

## Setup

Run **Kernel → Restart Kernel and Run All**. Use **`%matplotlib agg`**. Stdout should mention **UI version** when present.


In [1]:
%matplotlib agg
# Non-interactive backend; charts are embedded as PNG inside updatable HTML (no results Output widget).

import os

from ipywidgets import VBox, FloatSlider, Checkbox, HTML, Layout
from IPython.display import display, clear_output

from rate_allocator import allocate, build_interactive_report_html
from rate_allocator.adapters.yaml_loader import load_institutions_with_overrides

if os.path.basename(os.getcwd()) == 'notebooks':
    data_file = '../data/sample1.yaml'
else:
    data_file = 'data/sample1.yaml'

base_institutions = load_institutions_with_overrides(data_file, {})
print(f"Loaded {len(base_institutions)} institutions from {data_file}")
print("demo_ipywidgets UI version: 5 (src-backed report builder).")

Loaded 5 institutions from ../data/sample1.yaml
demo_ipywidgets UI version: 5 (src-backed report builder).


## Compact Control Panel

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
def _brief_constraints_label(inst):
    parts = []
    for tier in inst.tiers:
        for c in tier.constraints:
            parts.append(f"{c.type} ${c.cost:.2f}")
    return ", ".join(parts) if parts else "no modeled fees"


allocation_slider = FloatSlider(
    value=100_000,
    min=0,
    max=100_000,
    step=1_000,
    description='Total (MXN):',
    continuous_update=False,
    readout_format='.0f',
    layout=Layout(width='100%'),
)

institution_checkboxes = {}
for inst in base_institutions:
    hint = _brief_constraints_label(inst)
    institution_checkboxes[inst.name] = Checkbox(
        value=True,
        description=f"{inst.name} ({hint})",
        indent=False,
        layout=Layout(width='100%'),
    )

footnote = HTML(
    "<small>Use each <b>Institution</b> checkbox to include or exclude that provider from the optimization. "
    "Text in parentheses summarizes YAML fees or conditions (the allocator applies them when that institution is included). "
    "Headline <em>expected return</em> uses the selected horizon with compound gross return minus horizon fees, including estimated ISR over real interest (using the inflation proxy). "
    "Bank withholding is reported separately as cash-flow and is not subtracted in optimization net return. "
    "The chart below uses the same allocation with daily-style compounding (m=365) vs simple accrual.</small>"
)

control_panel = VBox(
    [
        HTML("<b>Institutions</b>"),
        *institution_checkboxes.values(),
        footnote,
        allocation_slider,
    ],
    layout=Layout(max_width='720px'),
)

display(control_panel)

## Results

Run **Restart Kernel and Run All**. Allocation results and control wiring live in **one code cell** below (`clear_output` + fresh `display_id` each run).


### Allocation results

Table and charts are created in the **next code cell** together with widget wiring. That cell calls `clear_output` first so reopening the notebook and running again does not stack old HTML. Use **Restart Kernel and Run All** after opening the file.


In [ ]:
import time
from IPython.display import clear_output, display, HTML

# One cell for results + wiring: clear this cell's outputs first (stops stacked HTML/widgets after reopen + Run All).
clear_output(wait=True)

RESULTS_DISPLAY_ID = f"rate-allocator-results-{time.time_ns()}"
display(HTML("<b>Allocation results</b>"))
alloc_results_handle = display(
    HTML("<p><i>Loading...</i></p>"),
    display_id=RESULTS_DISPLAY_ID,
)


def _push_results_html(fragment: str):
    alloc_results_handle.update(HTML(fragment))


def build_selected_institutions():
    return [name for name, checkbox in institution_checkboxes.items() if checkbox.value]


def run_allocation(_=None):
    selected_names = build_selected_institutions()

    if not selected_names:
        _push_results_html("<p>Select at least one institution.</p>")
        return

    all_institutions = load_institutions_with_overrides(data_file, {})
    institutions = [inst for inst in all_institutions if inst.name in selected_names]

    total = allocation_slider.value
    horizon_years = 1.0
    result = allocate(
        total=total,
        institutions=institutions,
        horizon_years=horizon_years,
        periods_per_year=365,
    )

    html_fragment = build_interactive_report_html(
        result,
        institutions,
        total=total,
        horizon_years=horizon_years,
        periods_per_year=365,
    )
    _push_results_html(html_fragment)


for checkbox in institution_checkboxes.values():
    checkbox.unobserve_all(name="value")
allocation_slider.unobserve_all(name="value")

for checkbox in institution_checkboxes.values():
    checkbox.observe(run_allocation, names="value")
allocation_slider.observe(run_allocation, names="value")

run_allocation()


## Notes

In [5]:
print(
    "Institution checkboxes include or exclude each provider from the optimization "
    "(YAML fees and conditions still apply when that institution is included; fees include estimated ISR over real interest, and withholding is shown separately)."
)

Institution checkboxes include or exclude each provider from the optimization (YAML fees and conditions still apply when that institution is included; fees include estimated ISR over real interest, and withholding is shown separately).


Constraint costs from YAML and estimated ISR over real interest are deducted from expected return over the selected horizon (see footnotes and fee columns in the output tables).

Net return is shown as separate nominal and real columns. Bank withholding is displayed as a separate cash-flow metric and is not subtracted in optimization net return.